# Energy Transition & Electric Mobility in Portugal - Cleaning

**Phase 2: Cleaning**

Transforms the raw REN JSON (Highcharts format: one time series per energy
source, 96 fifteen-minute readings per day) into a tidy daily-aggregate
table — one row per date + energy source.

In [1]:
import json
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd()
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", None)

## Load the raw REN data

Each entry is `{"date": ..., "data": {...}}`, where `data` holds the
Highcharts structure: `xAxis.categories` (96 time labels shared by every
series) and `series` (a list of energy sources, each with 96 MW readings).

In [2]:
with open(DATA_RAW / "ren_production_breakdown_raw.json") as f:
    ren_raw = json.load(f)

print(f"Loaded {len(ren_raw)} days")

Loaded 1826 days


## Convert one day into tidy rows

Power (MW) is converted to energy (MWh) by multiplying by 0.25 - each
reading represents a 15-minute interval, i.e. 0.25 hours.

This function treats every series the same way (Hydro, Solar, Consumption,
Import, Export, batteries...) without judging which ones count as
"renewable production" - that classification belongs to the analysis step
later, not to cleaning.

In [3]:
def process_ren_day(day_entry):
    """
    Convert one day of REN data into daily-aggregate rows — one row per
    energy source, with total energy, average/peak power, and peak time.
    """
    day_date = day_entry["date"]
    day_data = day_entry["data"]

    # The 96 time labels (00:00, 00:15, ..., 23:45) are shared by every series
    time_labels = day_data["xAxis"]["categories"]

    rows = []

    for series in day_data["series"]:
        source_name = series["name"]
        values = series["data"]  # list of 96 MW readings, may contain None

        # Drop missing readings before doing math on them — a day with a
        # few gaps shouldn't crash the whole pipeline
        valid_pairs = [
            (time_labels[i], v) for i, v in enumerate(values) if v is not None
        ]

        if not valid_pairs:
            # entire series missing for this day — skip it, don't fabricate a zero
            continue

        valid_times = [t for t, v in valid_pairs]
        valid_values = [v for t, v in valid_pairs]

        # Power (MW) -> Energy (MWh): each reading = 15 min = 0.25 h
        energia_mwh = sum(valid_values) * 0.25
        potencia_media_mw = sum(valid_values) / len(valid_values)
        potencia_pico_mw = max(valid_values)

        # time label matching the peak value (first occurrence if tied)
        peak_index = valid_values.index(potencia_pico_mw)
        hora_pico = valid_times[peak_index]

        rows.append(
            {
                "date": day_date,
                "source": source_name,
                "energia_mwh": round(energia_mwh, 2),
                "potencia_media_mw": round(potencia_media_mw, 2),
                "potencia_pico_mw": potencia_pico_mw,
                "hora_pico": hora_pico,
            }
        )

    return rows

## Run it over every day

In [4]:
all_rows = []
for day_entry in ren_raw:
    all_rows.extend(process_ren_day(day_entry))

df_ren_daily = pd.DataFrame(all_rows)

print(f"Total rows: {len(df_ren_daily)}")  # ~1097 days x up to 15 sources
print(f"Distinct sources: {df_ren_daily['source'].nunique()}")
df_ren_daily.head(10)

Total rows: 25022
Distinct sources: 15


,date,source,energia_mwh,potencia_media_mw,potencia_pico_mw,hora_pico
0,2021-10-07,Consumption + Storage,143389.62,5974.57,7023.2,19:45
1,2021-10-07,Consumption,138205.40,5758.56,7023.2,19:45
2,2021-10-07,Pumping,5184.23,216.01,1123.2,04:30
3,2021-10-07,Hydro,23394.03,974.75,3953.8,19:30
4,2021-10-07,Solar,6291.00,262.12,823.0,13:00
5,2021-10-07,Import,32900.53,1370.86,3415.9,14:45
6,2021-10-07,Export,6320.07,263.34,2266.4,07:00
7,2021-10-07,Wind,13554.92,564.79,1104.8,01:30
8,2021-10-07,Natural Gas,59085.00,2461.88,2842.6,07:30
9,2021-10-07,Other Thermal,660.80,27.53,29.0,17:45


## Classify sources and calculate daily renewable %

Pivot the long table (one row per source) into one row per date, then
compute the renewable share using only production sources - demand/flow
series (Consumption, Import, Export, Pumping, batteries) are excluded.

In [5]:
RENEWABLE_SOURCES = ["Hydro", "Solar", "Wind", "Biomass", "Wave"]
NON_RENEWABLE_SOURCES = ["Natural Gas", "Other Thermal", "Coal"]
# Everything else (Consumption, Import, Export, Pumping, battery flows) is
# demand/flow, not production - excluded from the renewable % calculation

# Pivot: one row per date, one column per source, values = energia_mwh
df_wide = df_ren_daily.pivot(index="date", columns="source", values="energia_mwh")

df_wide["renovavel_mwh"] = df_wide[RENEWABLE_SOURCES].sum(axis=1)
df_wide["nao_renovavel_mwh"] = df_wide[NON_RENEWABLE_SOURCES].sum(axis=1)
df_wide["producao_total_mwh"] = df_wide["renovavel_mwh"] + df_wide["nao_renovavel_mwh"]
df_wide["pct_renovavel"] = (
    df_wide["renovavel_mwh"] / df_wide["producao_total_mwh"] * 100
).round(1)

print(f"Date range: {df_wide.index.min()} to {df_wide.index.max()}")
print(f"Average renewable %: {df_wide['pct_renovavel'].mean():.1f}%")
df_wide[["renovavel_mwh", "nao_renovavel_mwh", "pct_renovavel"]].head(10)

Date range: 2021-10-07 to 2026-10-07
Average renewable %: 77.6%


source,renovavel_mwh,nao_renovavel_mwh,pct_renovavel
date,,,
2021-10-07,51988.68,65095.00,44.4
2021-10-08,54492.04,63281.93,46.3
2021-10-09,47713.87,28461.85,62.6
2021-10-10,68481.27,12911.55,84.1
2021-10-11,74599.02,33148.62,69.2
2021-10-12,54997.89,32406.29,62.9
2021-10-13,67976.64,47319.72,59.0
2021-10-14,57160.77,65746.92,46.5
2021-10-15,60699.69,68808.58,46.9


In [6]:
# Check how much Import contributes relative to total production —
# if it's a similarly large share as the 26.3% cited nationally for
# Sept 2025, that supports import-exclusion as the reason our %
# runs higher than the official (production + imports) methodology
df_wide["import_share_pct"] = (
    df_wide["Import"] / (df_wide["producao_total_mwh"] + df_wide["Import"]) * 100
).round(1)

print(
    f"Average import share of total supply: {df_wide['import_share_pct'].mean():.1f}%"
)

# Alternative metric: renewable % of production + imports, treating all
# imported electricity as non-renewable (a conservative worst case) —
# this should land closer to the official 75-80% range if the hypothesis is right
df_wide["pct_renovavel_com_import"] = (
    df_wide["renovavel_mwh"] / (df_wide["producao_total_mwh"] + df_wide["Import"]) * 100
).round(1)

print(
    f"Average renewable % (imports counted as non-renewable): {df_wide['pct_renovavel_com_import'].mean():.1f}%"
)

Average import share of total supply: 21.6%
Average renewable % (imports counted as non-renewable): 61.1%


## Save the daily production table

Two renewable % columns are kept intentionally:
- `pct_renovavel`: share of domestic production only (Import/Export excluded)
- `pct_renovavel_com_import`: conservative alternative, treating all imported
  electricity as non-renewable - closer to how official national statistics
  (which include the import balance) are usually reported

Validated against APREN/DGEG published figures: the official rolling-year
value (75.5%) falls between these two, which is the expected result if the
gap is explained by import treatment rather than a calculation error.

In [7]:
df_wide.to_csv(DATA_PROCESSED / "ren_daily_production.csv")
print(f"Saved {len(df_wide)} rows to {DATA_PROCESSED / 'ren_daily_production.csv'}")

Saved 1826 rows to c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\processed\ren_daily_production.csv


## E-REDES - electric mobility consumption

Already a flat table from the extraction (server-side aggregation), so this
needs light cleaning: build a real date column, convert kWh to MWh for
consistency with the REN table, and re-confirm district coverage.

In [8]:
with open(DATA_RAW / "eredes_mobility_grouped_raw.json") as f:
    eredes_raw = json.load(f)

df_eredes = pd.DataFrame(eredes_raw)
print(f"Loaded {len(df_eredes)} rows")
df_eredes.head()

Loaded 3336 rows


,ano,mes,distrito,concelho,total_energia_kwh
0,2025,05,Aveiro,Albergaria-a-Velha,63111.00
1,2025,05,Aveiro,Anadia,17954.91
2,2025,05,Aveiro,Arouca,3050.21
3,2025,05,Aveiro,Aveiro,134610.62
4,2025,05,Aveiro,Castelo de Paiva,3675.95


In [9]:
# ano/mes arrive as text (e.g. "2025", "05") — combine into a real date,
# using day 1 of the month since this data has monthly granularity
df_eredes["date"] = pd.to_datetime(df_eredes["ano"] + "-" + df_eredes["mes"] + "-01")

# kWh -> MWh, to match the unit used in the REN production table
df_eredes["energia_mwh"] = (df_eredes["total_energia_kwh"] / 1000).round(3)

# Keep only the clean columns going forward
df_eredes_clean = (
    df_eredes[["date", "distrito", "concelho", "energia_mwh"]]
    .sort_values(["date", "distrito", "concelho"])
    .reset_index(drop=True)
)

print(
    f"Distinct distritos: {df_eredes_clean['distrito'].nunique()}"
)  # sanity check: expect 18
print(f"Date range: {df_eredes_clean['date'].min()} to {df_eredes_clean['date'].max()}")
df_eredes_clean.head()

Distinct distritos: 18
Date range: 2025-05-01 00:00:00 to 2026-04-01 00:00:00


,date,distrito,concelho,energia_mwh
0,2025-05-01,Aveiro,Albergaria-a-Velha,63.111
1,2025-05-01,Aveiro,Anadia,17.955
2,2025-05-01,Aveiro,Arouca,3.050
3,2025-05-01,Aveiro,Aveiro,134.611
4,2025-05-01,Aveiro,Castelo de Paiva,3.676


In [10]:
df_eredes_clean.to_csv(DATA_PROCESSED / "eredes_mobility_monthly.csv", index=False)
print(
    f"Saved {len(df_eredes_clean)} rows to {DATA_PROCESSED / 'eredes_mobility_monthly.csv'}"
)

Saved 3336 rows to c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\processed\eredes_mobility_monthly.csv


## Market prices - load and clean

24 hourly readings per day, two series (PT, ES). Daily average, min and max
are kept - min/max matter here because low prices often coincide with high
renewable output, which is exactly what the correlation step will test.
A `price_diverges` flag also marks days where PT and ES prices differ
(MIBEL market splitting due to grid congestion between the two countries).

Known gap: the most recent ~92 days have no price data yet - REN publishes
market prices with a delay. Not a bug, just a limitation to keep in mind
during analysis.

In [11]:
with open(DATA_RAW / "ren_market_prices_raw.json") as f:
    market_price_raw = json.load(f)

print(f"Loaded {len(market_price_raw)} days")

Loaded 1727 days


In [12]:
def process_price_day(day_entry):
    """Convert one day of REN market price data into a single summary row."""
    day_date = day_entry["date"]
    day_data = day_entry["data"]

    row = {"date": day_date}

    for series in day_data["series"]:
        country = series["name"]  # "PT" or "ES"
        values = [v for v in series["data"] if v is not None]

        if not values:
            continue

        row[f"price_avg_{country}"] = round(sum(values) / len(values), 2)
        row[f"price_min_{country}"] = min(values)
        row[f"price_max_{country}"] = max(values)

    return row


price_rows = [process_price_day(entry) for entry in market_price_raw]
df_prices = pd.DataFrame(price_rows)
df_prices["date"] = pd.to_datetime(df_prices["date"])

df_prices["price_diverges"] = (
    df_prices["price_avg_PT"] - df_prices["price_avg_ES"]
).abs() > 1.0

print(f"Rows: {len(df_prices)}")
print(
    f"Days with market split (PT != ES): {df_prices['price_diverges'].sum()} ({df_prices['price_diverges'].mean() * 100:.1f}%)"
)
df_prices.head()

Rows: 1727


Days with market split (PT != ES): 274 (15.9%)


,date,price_avg_PT,price_min_PT,price_max_PT,price_avg_ES,price_min_ES,price_max_ES,price_diverges
0,2021-10-07,288.53,249.08,319.03,288.53,249.08,319.03,False
1,2021-10-08,229.20,188.60,290.00,229.20,188.60,290.00,False
2,2021-10-09,226.21,176.92,254.45,226.21,176.92,254.45,False
3,2021-10-10,175.91,135.07,251.60,175.64,135.07,251.60,False
4,2021-10-11,184.48,125.01,241.27,184.48,125.01,241.27,False


In [13]:
df_prices.to_csv(DATA_PROCESSED / "ren_market_prices_daily.csv", index=False)
print(
    f"Saved {len(df_prices)} rows to {DATA_PROCESSED / 'ren_market_prices_daily.csv'}"
)

Saved 1727 rows to c:\Users\joaof\Desktop\Projects_db\Python\PT-Energy-Transition-Analytics\data\processed\ren_market_prices_daily.csv


## Next: 03_EDA.ipynb